# 机构调研策略 - 回测分析

本notebook实现研报中三种策略的回测:

1. **事件驱动策略** - 年化超额收益 >12%
2. **定期选股策略** - 年化超额收益 >18%
3. **行业轮动策略** - 年化超额收益 >7%

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source.data_loader import DataLoader
from source.data_preprocessor import DataPreprocessor
from source.backtest import BacktestEngine
from source.strategies.event_driven import EventDrivenStrategy
from source.strategies.regular_stock import RegularStockStrategy
from source.strategies.industry_rotation import IndustryRotationStrategy

## 1. 初始化

In [ ]:
# 初始化组件
dl = DataLoader()
preprocessor = DataPreprocessor()
backtest_engine = BacktestEngine(initial_capital=10000000, commission_rate=0.001)

event_strategy = EventDrivenStrategy()
regular_strategy = RegularStockStrategy()
industry_strategy = IndustryRotationStrategy()

print("所有组件初始化成功")

## 2. 加载数据

In [ ]:
# 获取指数数据
index_data = dl.get_index_data(index_code='000985.SH', start_date='20150101', end_date='20210228')
print(f"基准指数数据: {len(index_data)} 条")

# 获取股票数据(示例:获取部分股票数据用于演示)
print("注意: 完整回测需要全市场股票数据")

## 3. 策略回测框架

**注意**: 由于完整的机构调研数据需要商业终端获取,以下提供完整的回测框架代码。当数据可用时,可直接运行。

In [ ]:
def run_full_backtest_example():
    """
    完整回测示例函数
    
    参数说明:
    - survey_df: 机构调研数据,包含 ts_code, survey_date, institutions_count
    - price_df: 股票价格数据,包含 ts_code, trade_date, open, high, low, close, volume
    - benchmark_df: 基准指数数据,包含 trade_date, close
    """
    print("=" * 60)
    print("机构调研策略完整回测框架")
    print("=" * 60)
    
    # 回测参数设置(参考研报)
    lookback_days = 60  # 回看天数
    holding_days = 100  # 持仓天数
    threshold = 50     # 机构调研次数阈值
    
    print(f"\n策略参数:")
    print(f"  回看天数: {lookback_days}")
    print(f"  持仓天数: {holding_days}")
    print(f"  调研阈值: {threshold}")
    
    # 事件驱动策略回测
    print("\n1. 事件驱动策略回测...")
    event_results = event_strategy.run_strategy2(survey_df, price_df)
    if event_results:
        print(f"   年化收益率: {event_results.get('annual_return', 0):.2%}")
        print(f"   年化超额收益: {event_results.get('annualized_excess_return', 0):.2%}")
        print(f"   信息比率: {event_results.get('information_ratio', 0):.2f}")
    
    # 定期选股策略回测
    print("\n2. 定期选股策略回测...")
    regular_results = regular_strategy.run_typical_strategy(survey_df, price_df, num_stocks=20)
    if regular_results:
        print(f"   年化收益率: {regular_results.get('annual_return', 0):.2%}")
        print(f"   年化超额收益: {regular_results.get('annualized_excess_return', 0):.2%}")
        print(f"   信息比率: {regular_results.get('information_ratio', 0):.2f}")
    
    # 行业轮动策略回测
    print("\n3. 行业轮动策略回测...")
    industry_results = industry_strategy.run_typical_strategy(survey_df, price_df, num_industries=5)
    if industry_results:
        print(f"   年化收益率: {industry_results.get('annual_return', 0):.2%}")
        print(f"   年化超额收益: {industry_results.get('annualized_excess_return', 0):.2%}")
        print(f"   信息比率: {industry_results.get('information_ratio', 0):.2f}")
    
    return {
        'event': event_results,
        'regular': regular_results,
        'industry': industry_results
    }

print("回测框架已定义,需要补充机构调研数据后运行")

## 4. 研报复现要点

### 事件驱动策略 (年化超额收益 >12%)
- **策略1**: 最近1日股票被机构调研次数>50时买入,持有200天后卖出
- **策略2**: 最近60日股票总计被机构调研次数>50时买入,持有100天后卖出
- **回测区间**: 2015年1月1日至2021年2月28日

### 定期选股策略 (年化超额收益 >18%)
- **参数**: 回看天数120日,周度调仓,持股数20只
- **选股方式**: 按过去一段时间调研次数排序,持有机构关注度最高的股票

### 行业轮动策略 (年化超额收益 >7%)
- **Z-score方法**: 
  - 计算行业内个股平均被调研次数
  - 计算Z-score判断行业关注度
  - Z-score>1: 100%仓位; 0<Z-score<=1: 50%仓位; Z-score<=0: 空仓

In [ ]:
# 参数优化示例(伪代码,需要实际数据)
print("=" * 60)
print("参数优化框架")
print("=" * 60)

# 事件驱动策略参数优化
event_params = {
    'lookback_days_list': [1, 5, 10, 20, 40, 60],
    'holding_days_list': [20, 40, 60, 100, 200],
    'threshold_list': [20, 50, 80, 100, 200, 300]
}
print("\n事件驱动策略参数范围:")
for k, v in event_params.items():
    print(f"  {k}: {v}")

# 定期选股策略参数优化
regular_params = {
    'lookback_days_list': [10, 20, 40, 60, 120],
    'rebalance_freq_list': ['weekly', 'monthly', 'quarterly', 'semi-annual'],
    'num_stocks_list': [20, 50, 80, 100, 200]
}
print("\n定期选股策略参数范围:")
for k, v in regular_params.items():
    print(f"  {k}: {v}")

## 数据需求说明

**缺少的数据**:

1. **机构调研数据** (核心数据): 
   - 来源: Wind商业数据库 ASHAREINSTITUTIONALACTIVITY
   - 包含: 调研日期、调研机构数、股票代码等信息
   - 时间范围: 2012年至今
   
2. **股票价格数据**:
   - 可通过tushare、akshare等免费接口获取
   - 需要包含: 日线OHLCV数据
   
**下一步**:
- 请补充机构调研数据到 `output/data/survey_data.csv`
- 运行完整回测代码